# Mapear os TermpointToGnodeB (5G-5G) verificando se existe o TermpointToGNB (4G-5G)
# Verificar as NRCellRelation criadas e comparar com a GUtranCellRelation por cellId

#### Atualizado em 07/05/2025:
#####   - Missing TermpointToGNB: 
##### Ajustado para copiar e colar na macro removendo a coluna Ancora
##### Realizada a ordenação dos dados para rodar na macro sem precisar ordenar no excel
##### Incluido o GUtranFreqRelation no Audit GUtranCellRelation (10/07/2025)

In [ ]:
from __future__ import annotations
# PSCell rules embedded from MoB Tool Box (2026-11-09), active site_key_mode=cell.
# No runtime dependency on MoB or its configuration.
import pandas as pd
import numpy as np
from pathlib import Path
from types import SimpleNamespace
from math import asin, cos, radians, sin, sqrt
from typing import Tuple
_LTE_PREFIX = 1
_NR_PREFIX = 2
_SITE_CODE_LEN = 3
_SUFIXO_MAX_DIGITOS = 3
EARTH_RADIUS_KM = 6371.0088
def hotel_split_enabled():
    return True
def _uf_from_node(node: pd.Series) -> pd.Series:
    """UF pelo sufixo do nome do no ('ABCD1_XX' -> 'XX')."""
    return node.astype(str).str[-2:]

def _mid(series: pd.Series, start: int) -> pd.Series:
    return series.astype(str).str[start:start + _SITE_CODE_LEN]

def _nome_encaixa(cell: pd.Series, prefixo: int) -> pd.Series:
    resto = cell.astype(str).str[prefixo + _SITE_CODE_LEN:]
    return resto.str.fullmatch(rf"\d{{1,{_SUFIXO_MAX_DIGITOS}}}").fillna(False)

def host_key(node: pd.Series, uf: pd.Series | None = None) -> pd.Series:
    """UF + eNB[1:4] — o site do NO hospedeiro."""
    uf = _uf_from_node(node) if uf is None else uf.astype(str)
    return uf + _mid(node, _LTE_PREFIX)

def cell_key_lte(cell: pd.Series, uf: pd.Series) -> pd.Series:
    """UF + CELL[1:4] — o site FISICO de uma celula LTE."""
    return uf.astype(str) + _mid(cell, _LTE_PREFIX)

def cell_key_nr(cell: pd.Series, uf: pd.Series) -> pd.Series:
    """UF + CELL[2:5] — o site FISICO de uma celula NR."""
    return uf.astype(str) + _mid(cell, _NR_PREFIX)

def sector(cell: pd.Series) -> pd.Series:
    """Setor pelo ultimo caractere do nome: 4/5/6 e 7/8/9 repetem 1/2/3.

    Nao-digito vira 0 — os chamadores ja descartam setor 0.
    """
    last = cell.astype(str).str[-1]
    num = pd.to_numeric(last, errors="coerce")
    # `0` fica 0, como no mapeamento historico ([4..9] -> [1,2,3,1,2,3], que
    # deixava 0 intacto). Sem esta guarda ((0-1) % 3) + 1 daria 3.
    setor = ((num - 1) % 3) + 1
    setor = setor.where(num >= 1, 0)
    return setor.fillna(0).astype(int)

def site_lte(cell: pd.Series, node: pd.Series, uf: pd.Series | None = None) -> pd.Series:
    """Site de uma celula LTE, conforme o modo ativo."""
    uf = _uf_from_node(node) if uf is None else uf
    hospedeiro = host_key(node, uf)
    if hotel_split_enabled():
        return cell_key_lte(cell, uf).where(_nome_encaixa(cell, _LTE_PREFIX), hospedeiro)
    return hospedeiro

def site_nr(cell: pd.Series, node: pd.Series, uf: pd.Series | None = None) -> pd.Series:
    """Site de uma celula NR, conforme o modo ativo."""
    uf = _uf_from_node(node) if uf is None else uf
    hospedeiro = host_key(node, uf)
    if hotel_split_enabled():
        return cell_key_nr(cell, uf).where(_nome_encaixa(cell, _NR_PREFIX), hospedeiro)
    return hospedeiro
sk = SimpleNamespace(site_lte=site_lte, site_nr=site_nr, sector=sector, cell_key_lte=cell_key_lte, hotel_split_enabled=hotel_split_enabled)
def haversine_km(src_coords: Tuple[float, float], tgt_coords: Tuple[float, float]) -> float:
    """Returns the great-circle distance between two lat/lon pairs in kilometers."""
    src_lat, src_lon = src_coords
    tgt_lat, tgt_lon = tgt_coords

    lat1 = radians(src_lat)
    lon1 = radians(src_lon)
    lat2 = radians(tgt_lat)
    lon2 = radians(tgt_lon)

    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1

    a = sin(delta_lat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(delta_lon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return EARTH_RADIUS_KM * c
def _load_coordenadas(onedrive_dir: Path) -> pd.DataFrame:
    """Coordenada por chave de site, do cadastro LTE.

    Mesma regra do `_cosite_lookup` do `mob_distancias` (fix de 02/09/2026):

    - a chave do NO continua existindo, porque o relatorio de TermPoint (MO de
      nivel de no) consulta por ela;
    - HOSPEDE DE HOTEL NAO DEFINE A COORDENADA DO NO. Num Hotel de BTS as
      celulas do hospede moram no no do hospedeiro, e o `drop_duplicates`
      antigo pegava QUALQUER linha da chave — devolvia coordenada plausivel,
      quilometros fora, sem erro nem buraco. Para a chave do no valem as
      celulas cujo nome CONCORDA com ele; no que so hospeda cai no conjunto
      completo e nao perde entrada;
    - o hospede ganha entrada sob a chave FISICA dele, que e a que o resto do
      Tool Box usa em modo `cell`.

    Em modo `enb` a chave fisica e a do no, entao o resultado e o de sempre.
    """
    coord_path = onedrive_dir / "EUtranCell_TDD_FDD.csv"
    if not coord_path.exists():
        print(f"[!] Coordinates file not found: {coord_path}")
        return pd.DataFrame(columns=["Site", "Latitude", "Longitude"])

    df = pd.read_csv(coord_path, encoding="ISO-8859-1",
                     usecols=["UF", "eNB", "CELL", "Latitude", "Longitude"])
    df = df.dropna(subset=["eNB", "CELL", "Latitude", "Longitude"])
    df["Site"] = df["UF"].astype(str) + df["eNB"].astype(str).str[1:4]
    df["_fisico"] = sk.cell_key_lte(df["CELL"].astype(str), df["UF"].astype(str))

    proprias = df[df["_fisico"] == df["Site"]].drop_duplicates(subset="Site")
    resto = df.drop_duplicates(subset="Site")
    tabela = pd.concat([proprias, resto]).drop_duplicates(subset="Site", keep="first")

    # Nunca sobrescreve uma entrada de no ja existente.
    hospedes = df[df["_fisico"] != df["Site"]].drop_duplicates(subset="_fisico")
    hospedes = hospedes[~hospedes["_fisico"].isin(set(tabela["Site"]))].copy()
    hospedes["Site"] = hospedes["_fisico"]

    out = pd.concat([tabela, hospedes])
    out = out[["Site", "Latitude", "Longitude"]].drop_duplicates(subset="Site")

    # O cadastro LTE nao cobre site 5G SEM co-site 4G — e o hospede de Hotel de
    # BTS costuma ser exatamente isso (indoor/DAS pendurado num no de rua).
    # Sem esta etapa a linha perderia a distancia ao ganhar a chave certa, o
    # que seria trocar um numero errado por um campo vazio. Medido no cluster
    # de 08/09/2026: 29 linhas, todas de um mesmo no hospedeiro.
    nr = _load_coordenadas_nr(onedrive_dir)
    if not nr.empty:
        nr = nr[~nr["Site"].isin(set(out["Site"]))]
        out = pd.concat([out, nr]).drop_duplicates(subset="Site")
    return out

def _load_coordenadas_nr(onedrive_dir: Path) -> pd.DataFrame:
    """Coordenada por chave de site, do cadastro 5G. So preenche o que falta.

    A chave sai de `sk.site_nr`, entao em modo `enb` ela e a chave do NO — que
    o cadastro LTE ja cobre — e nada e acrescentado.
    """
    path = onedrive_dir / "cellref_5g.xlsx"
    if not path.exists():
        return pd.DataFrame(columns=["Site", "Latitude", "Longitude"])
    try:
        df = pd.read_excel(path, usecols=["UF", "SiteID", "Cellname",
                                          "Latitude", "Longitude"])
    except Exception as exc:                       # cadastro fora do formato
        print(f"[!] cellref_5g nao lido ({exc}); distancia so pelo co-site 4G")
        return pd.DataFrame(columns=["Site", "Latitude", "Longitude"])

    df = df.dropna(subset=["SiteID", "Cellname", "Latitude", "Longitude"])
    if df.empty:
        return pd.DataFrame(columns=["Site", "Latitude", "Longitude"])
    df["Site"] = sk.site_nr(df["Cellname"].astype(str), df["SiteID"].astype(str),
                            df["UF"].astype(str))
    return df[["Site", "Latitude", "Longitude"]].drop_duplicates(subset="Site")

def _calculate_distance(row) -> float:
    """Calculates distance in km between src and tgt using haversine."""
    try:
        if pd.isna(row["Latitude_src"]) or pd.isna(row["Longitude_src"]) or \
           pd.isna(row["Latitude_tgt"]) or pd.isna(row["Longitude_tgt"]):
            return None
        src_coords = (row["Latitude_src"], row["Longitude_src"])
        tgt_coords = (row["Latitude_tgt"], row["Longitude_tgt"])
        return round(haversine_km(src_coords, tgt_coords), 2)
    except Exception:
        return None

def _add_distance(df: pd.DataFrame, coordenadas: pd.DataFrame,
                  col_site_src: str, col_site_tgt: str) -> pd.DataFrame:
    """Adds Distance column by computing haversine between src and tgt sites."""
    df = pd.merge(df, coordenadas, left_on=col_site_src, right_on="Site", how="left") \
           .rename(columns={"Latitude": "Latitude_src", "Longitude": "Longitude_src"})
    df.drop(columns=["Site"], inplace=True, errors="ignore")

    df = pd.merge(df, coordenadas, left_on=col_site_tgt, right_on="Site", how="left") \
           .rename(columns={"Latitude": "Latitude_tgt", "Longitude": "Longitude_tgt"})
    df.drop(columns=["Site"], inplace=True, errors="ignore")

    df["Distance"] = df.apply(_calculate_distance, axis=1)
    df.drop(columns=[col_site_src, col_site_tgt,
                     "Latitude_src", "Longitude_src",
                     "Latitude_tgt", "Longitude_tgt"], inplace=True, errors="ignore")
    df.drop_duplicates(inplace=True)
    return df

def _gfreq_cell_column(path: Path) -> str:
    """Nome da coluna da celula 4G no GUtranFreqRelation.txt.

    O pos-processamento grava `EUtranCellFDD`, mas dumps antigos/TDD podem trazer
    outro nome — mesma tolerancia do `data_loader.load_gutrans_freqrelation()`.
    """
    try:
        header = pd.read_csv(path, sep="\t", nrows=0, encoding="ISO-8859-1")
    except Exception:
        return ""
    for candidate in ("EUtranCellFDD", "EUtranCellTDD", "EUtranCell"):
        if candidate in header.columns:
            return candidate
    return ""
coordenadas = _load_coordenadas(Path("C:/Camilo"))

from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

In [ ]:
# Pegando os termpoints 5G--5G

import pandas as pd

arquivo_TermpointToGNodeB= 'C:/Camilo/' + '5G_TermpointToGNodeB'+ '_'+str(semana_atual)+'.txt'

# Abrindo o MO TermpointToGNode com apenas as colunas necessarias
TermpointToGNodeB= pd.read_csv(arquivo_TermpointToGNodeB, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'gNB', 'Ancora', 'operationalState']
                              ,low_memory=False)

# Filtrando as linhas onde 'gNB' ou 'Ancora' não começam com 'T'
TermpointToGNodeB = TermpointToGNodeB[(TermpointToGNodeB['gNB'].str[0] != 'T') | (TermpointToGNodeB['Ancora'].str[0] != 'T')]

TermpointToGNodeB['Site']= TermpointToGNodeB['gNB'].str[1:4] + '_' + TermpointToGNodeB['UF']

TermpointToGNodeB.head()

# Criando uma source 4G para comparar com o TermpointToGNB

In [ ]:
# Para criar a source vamos usar o arquivo EUtranCell_TDD_FDD que contem a lista de todos os sites da rede
EutranCell= pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.csv', low_memory=False,encoding='ISO-8859-1', 
                        usecols= ['UF', 'MUNICIPIO','eNB', 'CELL','cellId','administrativeState', 'earfcndl', 'CGI', 'Tipo'])

# Criando uma copia do EUtranCell para usarmos na identificação das GutranCellRelations (problema 2)
setores_lte= EutranCell.copy()
setores_lte.rename(columns={'Tipo': 'Tipo_Freq'},inplace=True)
EutranCell['Site']= EutranCell['eNB'].str[1:4] + '_' + EutranCell['UF']
# EutranCell= EutranCell[EutranCell['administrativeState']=='UNLOCKED']
EutranCell.drop(columns=['UF', 'CELL', 'cellId','MUNICIPIO', 'earfcndl', 'CGI', 'Tipo'], inplace=True)
EutranCell.rename(columns={'administrativeState': 'administrativeState_LTE'},inplace=True)
EutranCell.drop_duplicates(inplace=True)
EutranCell.head()

In [ ]:
TermpointToGNodeB_par= pd.merge(TermpointToGNodeB, EutranCell, on='Site', how='left')

TermpointToGNodeB_par.dropna(subset=['eNB'],inplace=True)
TermpointToGNodeB_par.head()

In [ ]:
# Pegando os termpoints 4G--5G

import pandas as pd

arquivo_TermpointToGNB= 'C:/Camilo/' + '5G_TermpointToGNB'+ '_'+str(semana_atual)+'.txt'

# Abrindo o MO TermpointToGNode com apenas as colunas necessarias
TermpointToGNB= pd.read_csv(arquivo_TermpointToGNB, sep='\t', encoding='ISO-8859-1', usecols=['UF','eNB', 'Ancora', 'administrativeState', 'operationalState']
                              ,low_memory=False)

TermpointToGNB= TermpointToGNB[TermpointToGNB['Ancora'].str[0]!= 'T']

# TermpointToGNB['Site']= TermpointToGNB['eNB'].str[1:4] + '_' + TermpointToGNB['UF']

TermpointToGNB.rename(columns= {'administrativeState': 'administrativeState_GNB', 'operationalState': 'operationalState_GNB', 'eNB': 'eNB_GNB'},inplace=True)

TermpointToGNB.drop(columns='UF',inplace=True)

TermpointToGNB.head()

## Verificando se para todo par 5G--5G (TermpointToGNodeB) existe um par 4G--5G (TermpointToGNB)

In [ ]:
par_4g_5g= pd.merge(TermpointToGNodeB_par, TermpointToGNB, left_on=['eNB', 'Ancora'], right_on=['eNB_GNB', 'Ancora'], how='left')
par_4g_5g.drop_duplicates(inplace=True)
# par_4g_5g.to_excel('C:/Camilao/par_4g_5g.xlsx', index=None)

# Identificando os pares de NRCellRelation

In [ ]:
arquivo_nrcell= 'C:/Camilo/' + '5G_NRCellRelation'+ '_'+str(semana_atual)+'.txt'

# Abrindo o MO TermpointToGNode com apenas as colunas necessarias
nrcell= pd.read_csv(arquivo_nrcell, sep='\t', encoding='ISO-8859-1', usecols=['UF_src','eNB_src', 'NRCellCU', 'eNB_tgt', 'nRCellCUId'],low_memory=False)

nrcell['Setor_src']= nrcell['NRCellCU'].str[-1]
nrcell['Setor_tgt']= nrcell['nRCellCUId'].str[-1]
nrcell['par']= nrcell['eNB_src'].str[1:4] + '_' + nrcell['UF_src'] + '_' + nrcell['eNB_tgt'].str[1:4] + nrcell['eNB_tgt'].str[-3:] + '_' + nrcell['Setor_src'] + '_' + nrcell['Setor_tgt']

nrcell.dropna(subset='eNB_tgt',inplace=True)
nrcell['Site_src']= sk.site_nr(nrcell['NRCellCU'], nrcell['eNB_src'], nrcell['UF_src'])
nrcell.head()

# NOVO - Criando uma Lista Virtual 4G--5G para Compararmos com o GUtranCellRelation

In [ ]:
setores_lte['Setor_src']= setores_lte['CELL'].str[-1].replace(['4','5','6', '7', '8', '9'], ['1','2','3', '1','2','3'])
setores_lte['Site_src']= sk.site_lte(setores_lte['CELL'], setores_lte['eNB'], setores_lte['UF'])
setores_lte.head()

In [ ]:
nrcell_fit= nrcell.copy()
nrcell_fit.drop(columns=['UF_src','eNB_src', 'NRCellCU', 'par', 'Setor_tgt'],inplace=True)
nrcell_fit['Setor_src']= nrcell_fit['Setor_src'].replace(['4','5','6', '7', '8', '9'], ['1','2','3', '1','2','3'])
nrcell_fit.drop_duplicates(inplace=True)
nrcell_fit.head()

In [ ]:
lista_4g_5G_virtual= pd.merge(setores_lte, nrcell_fit, on=['Site_src', 'Setor_src'], how='inner')

print(lista_4g_5G_virtual.shape)
lista_4g_5G_virtual.drop_duplicates(inplace=True)
lista_4g_5G_virtual.head()

# Identificando os pares de GUtranCellRelation

In [ ]:
arquivo_GUtranCell= 'C:/Camilo/' + '5G_GUtranCellRelation'+ '_'+str(semana_atual)+'.txt'

# Abrindo o MO TermpointToGNode com apenas as colunas necessarias
GUtranCell= pd.read_csv(arquivo_GUtranCell, sep='\t', encoding='ISO-8859-1', usecols=['UF','eNB', 'EUtranCell', 'Neighbor', 'nRCellCUId']
                              ,low_memory=False)

GUtranCell['Setor_src']= GUtranCell['EUtranCell'].str[-1]
GUtranCell['Setor_tgt']= GUtranCell['nRCellCUId'].str[-1]
GUtranCell['par']= GUtranCell['eNB'].str[1:4] + '_' + GUtranCell['UF'] + '_' + GUtranCell['Neighbor'].str[1:4] + GUtranCell['Neighbor'].str[-3:] + '_' + \
GUtranCell['Setor_src'] + '_' + GUtranCell['Setor_tgt']

GUtranCell.head()
# GUtranCell.to_excel('C:/Camilao/GUtranCell.xlsx',index=None)

# Problema n° 1 - Verificando se existe TermpointToGnodeB (5G--5G) e nRCellRelation, mas nao existe o TermpointToGNB

In [ ]:
termpoint_com_nr= pd.merge(par_4g_5g, nrcell, left_on=['gNB', 'Ancora'], right_on=['eNB_src', 'eNB_tgt'], how='inner')
# termpoint_com_nr.to_excel('C:/Camilao/termpoint_com_nr.xlsx',index=None)

In [ ]:
# Filtrando apenas os casos onde existe o termpointToGnodeB e ele esta ENABLED
missing_termpointtognb= termpoint_com_nr[(termpoint_com_nr['eNB_GNB'].isna())]

# Selecionando apenas as colunas eNB (src) e Ancora (tgt)
missing_termpointtognb1= missing_termpointtognb[['eNB', 'Ancora']]
missing_termpointtognb1= missing_termpointtognb1.copy()

# Adicionando as colunas UF e Regional para utilizar na macro
missing_termpointtognb1['UF']= missing_termpointtognb1['eNB'].str[-2:]
missing_termpointtognb1['Regional']= missing_termpointtognb1['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])
missing_termpointtognb1= missing_termpointtognb1[['Regional', 'UF','eNB', 'Ancora']]
missing_termpointtognb1.shape

In [ ]:
# Utilizando o MO NRCellCU para obter as informações da Ancora 

arquivo_NRCellCU= 'C:/Camilo/' + '5G_NRCellCU'+ '_'+str(semana_atual)+'.txt'

NR= pd.read_csv(arquivo_NRCellCU, sep='\t', engine='python')

NR['gNB']= NR['eNB']

# Retirei a condição abaixo para pegar casos onde o Site 5G esta sem cursar trafego devido a falta de ancora 4G
# NR.dropna(subset= ['cellState'], inplace=True)

NR['Termpoint']= NR['pLMNId_mcc'].astype(int).astype(str) + NR['pLMNId_mnc'].astype(int).astype(str) + '-' + NR['gNBId'].astype(int).astype(str)

NR1= NR.copy()

NR1= NR1.loc[(~NR1['nRCellCUId'].str[0:2].isin(['5R', '5Z'])),:]

NR1= NR1[['gNB', 'gNBId', 'Termpoint']]

NR1.drop_duplicates(inplace=True)

NR1.head()

In [ ]:
# Pegando as informações de IPAddress

ipaddress= pd.read_csv('C:/Dumps/LTE_Total/AddressIPv4.txt', sep='\t', usecols=['eNB', 'Router', 'IP_address'])

# Separando os IPs de NR_X2 - Referentes ao TermpointToGNB
ipaddress= ipaddress[ipaddress['Router']=='NR_X2']
ipaddress.drop(columns='Router', inplace=True)
ipaddress.rename(columns={'eNB': 'gNB'},inplace=True)
ipaddress.head()

In [ ]:
# Juntando as informações de IP com o NRCellDU

NR2= pd.merge(NR1, ipaddress, on='gNB', how='inner')
NR2['PLMN']= "mcc="+NR2['Termpoint'].str[:3] + ',mnc=' + NR2['Termpoint'].str[3:5] + ',mnclength=2'
NR2.head()

In [ ]:
# Juntando as informações da Ancora com o missingtermpoint 

missing_termpointtognb2= pd.merge(missing_termpointtognb1, NR2, left_on='Ancora', right_on='gNB', how='inner')
missing_termpointtognb2.drop(columns='Ancora',inplace=True)
missing_termpointtognb2.sort_values(by=['Regional', 'UF', 'eNB', 'IP_address', 'gNB'], ascending= True)
missing_termpointtognb2.drop_duplicates(inplace=True)
missing_termpointtognb2.head()
missing_termpointtognb2['Site_src'] = missing_termpointtognb2['UF'] + missing_termpointtognb2['eNB'].str[1:4]
missing_termpointtognb2['Site_tgt'] = missing_termpointtognb2['gNB'].str[-2:] + missing_termpointtognb2['gNB'].str[1:4]
missing_termpointtognb2 = _add_distance(missing_termpointtognb2, coordenadas, 'Site_src', 'Site_tgt')


In [ ]:
missing_termpointtognb2.to_excel('C:/Vivo Reports/5G/5G Mobilidade/Missing_TermPointToGNB_PSCellChange.xlsx',index=None)
print('Done!')

# Problema n° 2 - Verificando se existe o nRCellRelation, mas nao existe o par de GUtranCellRelation

In [ ]:
lista_4g_5G_virtual.head()

In [ ]:
missing_gutrancellrelation= pd.merge(lista_4g_5G_virtual, GUtranCell, left_on=['UF','eNB', 'CELL', 'eNB_tgt', 'nRCellCUId'], 
                                     right_on=['UF','eNB', 'EUtranCell', 'Neighbor', 'nRCellCUId'], how='left')
missing_gutrancellrelation= missing_gutrancellrelation[missing_gutrancellrelation['Neighbor'].isna()]
missing_gutrancellrelation.drop_duplicates(inplace=True)
missing_gutrancellrelation.head()

In [ ]:
# preciso transformar o missing_gutrancellrelation no mesmo formato do lte abaixo para dar sequencia ao script sem esforço

missing_gutrancellrelation.drop(columns=['Setor_src_x', 'Site_src', 'Setor_src_y', 'Setor_tgt', 'par', 'EUtranCell', 'Neighbor'], inplace=True)

missing_gutrancellrelation.head()                                

In [ ]:
# Adicionando agora os setores NR (tgt)

arquivo_NRCellDU= 'C:/Camilo/' + '5G_NRCellDU'+ '_'+str(semana_atual)+'.txt'

NRCellDU= pd.read_csv(arquivo_NRCellDU, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'eNB', 'nRCellDUId', 'cellLocalId', 'ssbFrequency', 'nRPCI', 'bandList',
                                                                        'administrativeState', 'operationalState'])
NRCellDU1= NRCellDU[(NRCellDU['administrativeState']=='UNLOCKED') & (NRCellDU['operationalState']=='ENABLED')]

NRCellDU1= NRCellDU1.rename(columns={'eNB': 'gNB'})
NRCellDU2= NRCellDU1.loc[(~NRCellDU1['nRCellDUId'].str[0:2].isin(['5R', '5Z'])),:]

# Calculando physicalLayerCellIdGroup e physicalLayerSubCellId
NRCellDU2 = NRCellDU2.copy() # usado para evitar warnings

NRCellDU2['physicalLayerCellIdGroup'] = NRCellDU2['nRPCI'] // 3 # Divisão inteira por 3
NRCellDU2['physicalLayerSubCellId'] = NRCellDU2['nRPCI'] % 3 # Resto da divisão por 3

NRCellDU2['bandList']=  NRCellDU2['bandList'].astype(int).astype(str).str[-2:]

NRCellDU2.rename(columns={'bandList': 'band'},inplace=True)

NRCellDU2.head()

In [ ]:
arquivo_NRCellCU= 'C:/Camilo/' + '5G_NRCellCU'+ '_'+str(semana_atual)+'.txt'
NRCellCU= pd.read_csv(arquivo_NRCellCU, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'eNB', 'gNBId','pLMNId_mcc', 'pLMNId_mnc'])
NRCellCU.drop_duplicates(inplace=True)

NRCellCU.rename(columns={'eNB': 'gNB'}, inplace=True)
setores_nr= pd.merge(NRCellDU2, NRCellCU, on=['UF', 'gNB'], how= 'inner')
# setores_nr['gNB_aux']= setores_nr['gNB'].str[1:4] + setores_nr['gNB'].str[-3:] + '_' + setores_nr['nRCellDUId'].str[-1]
setores_nr.head()

In [ ]:
lte_nr= pd.merge(missing_gutrancellrelation, setores_nr, left_on=['eNB_tgt', 'nRCellCUId'], right_on=['gNB', 'nRCellDUId'],
                 how='inner', suffixes=['_lte', '_nr'])

lte_nr = lte_nr[(lte_nr['administrativeState_lte'] == 'UNLOCKED') & (lte_nr['administrativeState_nr'] == 'UNLOCKED')]
lte_nr['gNB_aux']= lte_nr['gNB']
lte_nr.rename(columns={'gNB_aux': 'Ancora', 'UF_lte':'UF'},inplace=True)

lte_nr.drop(columns= ['administrativeState_lte'], inplace=True)
lte_nr.head()

# Pegando a informação de GUtranSyncSignalFrequency no MO GUtranFreqRelation

In [ ]:
# GUtranFreqRelation is scoped to the source CELL, not just the eNB.
arquivo_GUtranFreqRelation = 'C:/Camilo/GUtranFreqRelation_' + str(semana_atual) + '.txt'
cell_col = _gfreq_cell_column(Path(arquivo_GUtranFreqRelation))
if not cell_col:
    raise ValueError('GUtranFreqRelation sem coluna de celula; associacao insegura')
GUtranFreqRelation = pd.read_csv(arquivo_GUtranFreqRelation, sep='\t', usecols=['UF', 'eNB', cell_col, 'GUtranSyncSignalFrequency', 'GUtranFrequency', 'GUtranFreqRelationId'], encoding='ISO-8859-1')
GUtranFreqRelation.rename(columns={cell_col: 'CELL', 'GUtranFrequency': 'ssbFrequency'}, inplace=True)
freq_keys = ['UF', 'eNB', 'CELL', 'ssbFrequency']
GUtranFreqRelation.drop_duplicates(inplace=True)
antes = len(GUtranFreqRelation)
GUtranFreqRelation = GUtranFreqRelation.drop_duplicates(subset=freq_keys, keep='first')
if len(GUtranFreqRelation) != antes:
    print(f'[!] {antes-len(GUtranFreqRelation)} GUtranFreqRelation repetidas por celula/frequencia; mantida a primeira, como no MoB')


In [ ]:
# Juntando as informações de missing neighbor com GUtranFreqRelation

lte_nr1= pd.merge(lte_nr, GUtranFreqRelation, on=freq_keys, how= 'left')
pd.set_option('display.max_columns',None)
# lte_nr1.rename(columns={'Tipo': 'Tipo_Freq'},inplace=True)


# Adicionando a informação de ExternalGNodeBFunction do MO ExternalGUtranCell

In [ ]:
import numpy as np

# pegando as informações do MO ExternalGUtranCell
ExternalGUtranCell= pd.read_csv('C:/Dumps/LTE_total/5G_ExternalGUtranCell.txt', sep='\t', usecols=(['UF', 'eNB', 'ExternalGNodeBFunction', 'eNB_tgt']))

ExternalGUtranCell.rename(columns={'eNB_tgt': 'gNB'}, inplace=True)

ExternalGUtranCell.drop_duplicates(inplace=True)
ExternalGUtranCell.head()

# adicionando as informações de ExternalGNodeBFunction e ExternalGUtranCell ao missing neighbor
lte_nr2= pd.merge(lte_nr1, ExternalGUtranCell, on=['UF', 'eNB', 'gNB'], how='left')

# Physical site and normalized sector, matching MoB in cell mode.
_site_src = sk.site_lte(lte_nr2['CELL'], lte_nr2['eNB'], lte_nr2['UF'])
_site_tgt = sk.site_nr(lte_nr2['nRCellDUId'], lte_nr2['gNB'], lte_nr2['gNB'].str[-2:])
condition_co_site = _site_src == _site_tgt
condition_co_setor = condition_co_site & (sk.sector(lte_nr2['CELL']) == sk.sector(lte_nr2['nRCellDUId']))
lte_nr2['Tipo'] = np.select([condition_co_setor, condition_co_site], ['co_setor', 'co_site'], default='inter_site')
lte_nr2['Regional']= lte_nr2['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

lte_nr2= lte_nr2.sort_values(by=['Regional','UF', 'eNB', 'CELL', 'gNB', 'nRCellDUId'], ascending=([True, True, True, True, True, True]))

lte_nr2.head()

In [ ]:
# Ordenando as colunas de acordo com a macro

colunas = [
'UF', 'MUNICIPIO', 'eNB', 'CELL', 'Ancora', 'Tipo','gNB', 'GUtranSyncSignalFrequency', 'ssbFrequency', 'band', 'gNBId', 'nRCellDUId', 'cellLocalId', 'nRPCI', 'physicalLayerCellIdGroup', 'physicalLayerSubCellId'
,'pLMNId_mcc','pLMNId_mnc', 'administrativeState_nr','ExternalGNodeBFunction','GUtranFreqRelationId', 'Regional', 'Tipo_Freq'
]



lte_nr2 = lte_nr2[colunas]
lte_nr2.drop_duplicates(inplace=True)
lte_nr2.head()

In [ ]:
lte_nr2 = lte_nr2.sort_values(by=['UF', 'MUNICIPIO', 'eNB', 'gNB', 'nRCellDUId', 'CELL'])
lte_nr2['Site_src'] = sk.site_lte(lte_nr2['CELL'], lte_nr2['eNB'], lte_nr2['UF'])
lte_nr2['Site_tgt'] = sk.site_nr(lte_nr2['nRCellDUId'], lte_nr2['gNB'])
lte_nr2 = _add_distance(lte_nr2, coordenadas, 'Site_src', 'Site_tgt')
lte_nr2.to_csv('C:/Vivo Reports/5G/5G Mobilidade/Missing_GUtranCellRelation_PSCellChange.csv', encoding='ISO-8859-1', index=False)


# Problema n° 3 - Verificar casos onde existe o TermpointToGNodeB, existe o nRCellRelation, mas nao existe o EUtranCellRelation

In [ ]:
arquivo_EUtranCellRel= 'C:/Camilo/' + '5G_EUtranCellRelation'+ '_'+str(semana_atual)+'.txt'

EUtranCellRel= pd.read_csv(arquivo_EUtranCellRel, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'eNB', 'CELL', 'ExternalENodeBFunction', 
                                                                                           'ExternalEUtranCell', 'eNB_tgt', 'CELL_tgt','CGI', 'earfcndl_tgt'])

EUtranCellRel['par']= EUtranCellRel['eNB'].str[1:4] + '_' + EUtranCellRel['UF'] + '_' + EUtranCellRel['eNB_tgt'].str[1:4] + EUtranCellRel['eNB_tgt'].str[-3:] + '_' + \
EUtranCellRel['CELL'].str[-1] + '_' + EUtranCellRel['CELL_tgt'].str[-1]
EUtranCellRel.rename(columns={'eNB': 'gNB', 'CELL': 'nRCell'},inplace=True)
EUtranCellRel.head()

### Comparando os pares de NRCellRelation com o EUtranCellRelation
### Primeiro vamos criar uma lista virtual de NRCellRelation com seu tgt 4G

In [ ]:
# nrcell são os nrcellrelation existentes na rede

nrcell.rename(columns={'eNB_src': 'gNB_src', 'eNB_tgt': 'gNB_tgt'},inplace=True)
# Modificar a coluna 'gNB_tgt' diretamente
nrcell['eNB_aux'] = sk.site_nr(nrcell['nRCellCUId'], nrcell['gNB_tgt']) + '_' + nrcell['Setor_tgt'].str[-3:]


In [ ]:
setores_lte['eNB_aux'] = sk.site_lte(setores_lte['CELL'], setores_lte['eNB'], setores_lte['UF']) + '_' + setores_lte['CELL'].str[-1]


In [ ]:
nrcell.head(1)

In [ ]:
# Juntando os dados de nrcell com os setores lte para criar a lista virtual 5g--4g que deve existir na rede
virtual_5g_4g= pd.merge(nrcell, setores_lte, on='eNB_aux', how='inner')
virtual_5g_4g.drop_duplicates(inplace=True)
virtual_5g_4g.rename(columns={'gNB_src': 'gNB'},inplace=True)
virtual_5g_4g['Regional']= virtual_5g_4g['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

virtual_5g_4g= virtual_5g_4g[['Regional','UF', 'MUNICIPIO', 'gNB', 'NRCellCU', 'eNB', 'CELL', 'cellId','earfcndl','CGI','administrativeState']]
virtual_5g_4g['Tipo'] = sk.site_nr(virtual_5g_4g['NRCellCU'], virtual_5g_4g['gNB'], virtual_5g_4g['UF']) == sk.site_lte(virtual_5g_4g['CELL'], virtual_5g_4g['eNB'], virtual_5g_4g['eNB'].str[-2:])
virtual_5g_4g['Tipo'] = virtual_5g_4g['Tipo'].replace({True: 'co_site', False: 'inter_site'})



In [ ]:
missing_5g_4g= pd.merge(virtual_5g_4g, EUtranCellRel, left_on=['UF', 'gNB', 'NRCellCU', 'eNB', 'CELL'] , right_on=['UF', 'gNB', 'nRCell', 'eNB_tgt', 'CELL_tgt'],
                        how= 'left')

missing_5g_4g= missing_5g_4g[missing_5g_4g['nRCell'].isna()]
missing_5g_4g.rename(columns={'CGI_x': 'CGI'},inplace=True)

missing_5g_4g['ExternalENodeBFunction']= 'auto' + missing_5g_4g['CGI'].str[:3] + '_' + missing_5g_4g['CGI'].str[3:5] + '_2' + '_' + \
missing_5g_4g['CGI'].str.split('-').str[1]

missing_5g_4g= missing_5g_4g[['Regional','UF','MUNICIPIO','gNB','NRCellCU','eNB','CELL','cellId','earfcndl','CGI','ExternalENodeBFunction','administrativeState','Tipo']]
missing_5g_4g.drop_duplicates(inplace=True)
missing_5g_4g_mob = missing_5g_4g.copy()


In [ ]:
missing_5g_4g['Site']= missing_5g_4g['gNB'].str[1:4] + missing_5g_4g['gNB'].str[-3:]
missing_5g_4g['Site_tgt']= missing_5g_4g['eNB'].str[1:4] + missing_5g_4g['eNB'].str[-3:]
missing_5g_4g.shape

In [ ]:
TermpointToGNodeB_par['Site_tgt']= TermpointToGNodeB_par['Ancora'].str[1:4] + TermpointToGNodeB_par['Ancora'].str[-3:]


In [ ]:
# Comparando a lista de vizinhas missing com a lista virtual de TermpointToGNodeB_par. Assim saberemos se essa relação é de fato missing

missing_5g_4g_1= pd.merge(missing_5g_4g, TermpointToGNodeB_par, on=['UF', 'Site', 'Site_tgt'], how='inner')

missing_5g_4g_1.rename(columns={'gNB_x': 'gNB', 'eNB_x': 'eNB'},inplace=True)

missing_5g_4g_1= missing_5g_4g_1[['Regional','UF','MUNICIPIO','gNB','NRCellCU','eNB','CELL','cellId','earfcndl','CGI','ExternalENodeBFunction','administrativeState','Tipo']]

missing_5g_4g_1.drop_duplicates(inplace=True)

missing_5g_4g_1.head()

In [ ]:
NRCellDU3= NRCellDU2[['gNB', 'nRCellDUId', 'cellLocalId']].copy()
NRCellDU3.rename(columns={'nRCellDUId': 'NRCellCU'}, inplace=True)
NRCellDU3.head()

In [ ]:
# Adicionando a informação de celllocalId no setor src para macro criar os .mos
missing_5g_4g_2= pd.merge(missing_5g_4g_1, NRCellDU3, on= ['gNB', 'NRCellCU'], how= 'inner')
missing_5g_4g_2['eNodeBId'] = missing_5g_4g_2['CGI'].str.split('-').str[1].astype('int64')
missing_5g_4g_2.head()

In [ ]:
# Adicionando a informação da externalENodeBFunction

ExternalENodeBFunction= pd.read_csv('C:/Camilo/5G_ExternalENodeBFunction.txt', usecols=['eNB', 'eNodeBId', 'externalENodeBFunctionId']
                                    , sep='\t')
ExternalENodeBFunction.rename(columns={'eNB': 'gNB', 'externalENodeBFunctionId': 'externalENodeBFunctionId_MO'}, inplace=True)
ExternalENodeBFunction.head()

In [ ]:
missing_5g_4g_3= pd.merge(missing_5g_4g_2, ExternalENodeBFunction, on= ['gNB', 'eNodeBId'], how= 'left')
missing_5g_4g_3.head()

In [ ]:
missing_5g_4g_3= missing_5g_4g_3[['Regional', 'UF', 'MUNICIPIO', 'gNB', 'NRCellCU', 'cellLocalId', 'eNB', 'CELL', 'cellId', 'earfcndl', 'CGI', 'ExternalENodeBFunction',
'externalENodeBFunctionId_MO','administrativeState', 'Tipo']]
missing_5g_4g_3.head()

In [ ]:
# Preserve configured external MO and source local ID for existing consumers.
# These enrichments must not filter the MoB candidate set or multiply pairs.
missing_5g_4g_mob = missing_5g_4g_mob.merge(NRCellDU3.drop_duplicates(), on=['gNB', 'NRCellCU'], how='left', validate='many_to_one')
missing_5g_4g_mob['eNodeBId'] = missing_5g_4g_mob['CGI'].str.split('-').str[1].astype('int64')
external_config = pd.read_csv('C:/Camilo/5G_ExternalENodeBFunction.txt', sep='\t', usecols=['eNB', 'eNodeBId', 'externalENodeBFunctionId', 'pLMNId_mcc', 'pLMNId_mnc'])
external_config.rename(columns={'eNB': 'gNB', 'externalENodeBFunctionId': 'externalENodeBFunctionId_MO'}, inplace=True)
missing_5g_4g_mob['pLMNId_mcc'] = missing_5g_4g_mob['CGI'].str.split('-').str[0].str[:3].astype('int64')
missing_5g_4g_mob['pLMNId_mnc'] = missing_5g_4g_mob['CGI'].str.split('-').str[0].str[3:].astype('int64')
external_keys = ['gNB', 'eNodeBId', 'pLMNId_mcc', 'pLMNId_mnc']
missing_5g_4g_mob = missing_5g_4g_mob.merge(external_config.drop_duplicates(), on=external_keys, how='left', validate='many_to_one')
missing_5g_4g_mob.drop(columns=['eNodeBId', 'pLMNId_mcc', 'pLMNId_mnc'], inplace=True)
missing_5g_4g_mob['Site_src'] = sk.site_nr(missing_5g_4g_mob['NRCellCU'], missing_5g_4g_mob['gNB'], missing_5g_4g_mob['UF'])
missing_5g_4g_mob['Site_tgt'] = sk.site_lte(missing_5g_4g_mob['CELL'], missing_5g_4g_mob['eNB'])
missing_5g_4g_mob = _add_distance(missing_5g_4g_mob, coordenadas, 'Site_src', 'Site_tgt')
missing_5g_4g_mob.to_excel('C:/Vivo Reports/5G/5G Mobilidade/Missing_EUtranCellRelation_5g_4g_PSCellChange.xlsx', index=False)


### Identificando os pares TermpointToGnodeB e TermpointToENodeB existentes

In [ ]:
# Usar o TermpointToGNodeB_par para comparar com o TermpointToENodeB e assim identificar TermpointToENodeB missing
# Depois verificar se existe a EUtranCellRelation 

In [ ]:
# TermpointToGNodeB_par é uma lista virtual dos pares 5G--4G existentes na rede, usada para comparar com o que existe configurado no TermpointToENodeB
TermpointToGNodeB_par= TermpointToGNodeB_par[TermpointToGNodeB_par['gNB'].str[0]!='T']


In [ ]:
# Abrindo o TermpointToENodeB para compararmos com a lista de pares virtual

arquivo_TermpointToENodeB= 'C:/Camilo/' + '5G_TermpointToENodeB'+ '_'+str(semana_atual)+'.txt'

# Abrindo o MO TermpointToGNode com apenas as colunas necessarias
TermpointToENodeB= pd.read_csv(arquivo_TermpointToENodeB, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'eNB', 'Ancora', 'operationalState']
                              ,low_memory=False)

TermpointToENodeB= TermpointToENodeB[TermpointToENodeB['eNB'].str[0]!='T']
TermpointToENodeB.rename(columns={'eNB': 'gNB', 'Ancora':'eNB_termpoint'},inplace=True)


In [ ]:
# Comparando o TermpointToENodeB (5G-4G) com a lista virtual para identificar os TermpointToENodeB missing

missing_TermpointToENodeB= pd.merge(TermpointToGNodeB_par, TermpointToENodeB, left_on=['UF', 'Ancora', 'eNB'],
                                   right_on=['UF', 'gNB', 'eNB_termpoint'], how='left', suffixes=['_par', '_term'])

missing_TermpointToENodeB= missing_TermpointToENodeB[missing_TermpointToENodeB['eNB_termpoint'].isna()]

missing_TermpointToENodeB= missing_TermpointToENodeB[['UF', 'Ancora', 'operationalState_par', 'eNB', 'administrativeState_LTE', 'eNB_termpoint']]

missing_TermpointToENodeB.rename(columns={'Ancora': 'gNB', 'eNB': 'Ancora'}, inplace=True)
missing_TermpointToENodeB['Regional']= missing_TermpointToENodeB['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

coluna = missing_TermpointToENodeB.pop('Regional')
missing_TermpointToENodeB.insert(0, 'Regional', coluna)
missing_TermpointToENodeB= missing_TermpointToENodeB[missing_TermpointToENodeB['gNB'].str[0]!='T']


In [ ]:
setores_lte_1= setores_lte.copy()
setores_lte_1= setores_lte_1[['MUNICIPIO','eNB', 'CGI']]
setores_lte_1['CGI']= setores_lte_1['CGI'].str.rsplit('-', n=1).str[0]
setores_lte_1.drop_duplicates(inplace=True)
setores_lte_1.head()

In [ ]:
missing_TermpointToENodeB_1= pd.merge(missing_TermpointToENodeB, setores_lte_1, left_on='Ancora', right_on='eNB')
missing_TermpointToENodeB_1[['Regional', 'UF','MUNICIPIO', 'gNB', 'operationalState_par', 'Ancora', 'administrativeState_LTE', 'CGI', 'eNB_termpoint']]
missing_TermpointToENodeB_1.to_excel('C:/Camilao/missing_TermpointToENodeB_PSCellChange.xlsx',index=None)
print("Done!!!")

# Problema n°4 - Verificar se existe o TermpointToEnodeB (5G-4G) e a EUtranCellRelation (5G-4G), mas nao existe a volta GUtranCellRelation (4G-5G)
####     Obs.: Se existe o TermpointToENodeB (5G-4G) é porque existe o TermpointToGNB (4G-5G), pois ele é criado automaticamente após a criação do TermpointToGNB

In [ ]:
# Vamos começar identificando os TermpointToEnodeB existentes para cruzar com as EUtranCellRelations

# Usando o dataframe TermpointToENodeB e EUtranCellRel no join
par_termpoint_eutrancellrel= pd.merge(TermpointToENodeB, EUtranCellRel, left_on=['UF', 'gNB', 'eNB_termpoint'], right_on=['UF', 'gNB', 'eNB_tgt'], how='inner')
par_termpoint_eutrancellrel.drop(columns=['eNB_termpoint', 'par'], inplace=True)
par_termpoint_eutrancellrel.head()

In [ ]:
GUtranCell.drop(columns='par', inplace=True)


In [ ]:
missing_volta= pd.merge(par_termpoint_eutrancellrel, GUtranCell, left_on=['UF', 'eNB_tgt','CELL_tgt', 'gNB', 'nRCell'], 
                        right_on=['UF','eNB','EUtranCell', 'Neighbor','nRCellCUId'], how='left')

missing_volta1= missing_volta[missing_volta['eNB'].isna()]
missing_volta1= missing_volta1.copy()
missing_volta1= missing_volta1[['UF', 'eNB_tgt', 'CELL_tgt', 'earfcndl_tgt', 'gNB','nRCell']]
missing_volta1.rename(columns={'eNB_tgt': 'eNB', 'CELL_tgt': 'CELL', 'earfcndl_tgt': 'earfcndl'},inplace=True)


In [ ]:
# Verificando se existe o TermpointToGNB (volta) atravez do cruzamento do missing com o TermpointToGNB
missing_volta2= pd.merge(missing_volta1, TermpointToGNB, left_on=['eNB', 'gNB'], right_on=['eNB_GNB', 'Ancora'],how= 'left')

missing_volta2.head()

In [ ]:
setores_lte1= setores_lte.copy() 
setores_lte1= setores_lte[['UF', 'MUNICIPIO','eNB', 'Tipo_Freq']]
setores_lte1= setores_lte1.copy()
setores_lte1.drop_duplicates(inplace=True)

missing_volta3= pd.merge(missing_volta2, setores_lte1, on=['UF', 'eNB'], how='left')
# coluna = missing_volta3.pop('MUNICIPIO')
# missing_volta3.insert(1, 'MUNICIPIO', coluna)
missing_volta3['Ancora']= ''
missing_volta3['Tipo'] = (sk.site_lte(missing_volta3['CELL'], missing_volta3['eNB'], missing_volta3['UF']) == sk.site_nr(missing_volta3['nRCell'], missing_volta3['gNB'])).replace({True: 'co_site', False: 'inter_site'})
missing_volta3= missing_volta3[['UF','MUNICIPIO','eNB','CELL', 'Ancora', 'Tipo', 'gNB', 'nRCell', 'Tipo_Freq']]
missing_volta3.rename(columns={'nRCell': 'nRCellDUId'},inplace=True)
missing_volta3.drop_duplicates(inplace=True)
missing_volta3.head()

In [ ]:
setores_nr.head()

In [ ]:
missing_volta4= pd.merge(missing_volta3, setores_nr, on=['UF', 'gNB', 'nRCellDUId'], how='left')
missing_volta4.drop_duplicates(inplace=True)
missing_volta4.head()                       

In [ ]:
GUtranFreqRelation.head()

In [ ]:
missing_volta5= pd.merge(missing_volta4, GUtranFreqRelation, on= freq_keys, how='left')
# missing_volta5.drop(columns='gNB_aux',inplace=True)

missing_volta5= missing_volta5[['UF','MUNICIPIO','eNB','CELL','Ancora','Tipo','gNB', 'GUtranSyncSignalFrequency','ssbFrequency','band','gNBId',
                                'nRCellDUId','cellLocalId','nRPCI', 'physicalLayerCellIdGroup','physicalLayerSubCellId','pLMNId_mcc','pLMNId_mnc',
                                'administrativeState', 'Tipo_Freq']]
missing_volta5.head()

In [ ]:
# Adicionando a external

missing_volta6= pd.merge(missing_volta5,ExternalGUtranCell, on=['UF', 'eNB', 'gNB'], how='left')
missing_volta6['Regional']= missing_volta6['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])
missing_volta6['Site_src'] = sk.site_lte(missing_volta6['CELL'], missing_volta6['eNB'], missing_volta6['UF'])
missing_volta6['Site_tgt'] = sk.site_nr(missing_volta6['nRCellDUId'], missing_volta6['gNB'])
missing_volta6 = _add_distance(missing_volta6, coordenadas, 'Site_src', 'Site_tgt')


In [ ]:
# Agora vamos verificar se as missing neighbor ja foram mapeadas no audit GUtranCellRelation_PSCellChange
lte_nr2.head()
lte_nr3= lte_nr2[['eNB', 'CELL', 'gNB', 'nRCellDUId']]
lte_nr3.head()

In [ ]:
import pandas as pd

# Colunas para comparação
cols = ['eNB', 'CELL', 'gNB', 'nRCellDUId']

# Realiza o merge com indicator para identificar as linhas que não têm correspondência
missing_volta7 = missing_volta6.merge(lte_nr3[cols], on=cols, how='left', indicator=True)

# Filtra as linhas onde a combinação dos valores das colunas não está no lte_nr3
missing_volta7 = missing_volta7[missing_volta7['_merge'] == 'left_only'].drop(columns=['_merge'])

colunas = [
    "UF",
    "MUNICIPIO",
    "eNB",
    "CELL",
    "Ancora",
    "Tipo",
    "gNB",
    "GUtranSyncSignalFrequency",
    "ssbFrequency",
    "band",
    "gNBId",
    "nRCellDUId",
    "cellLocalId",
    "nRPCI",
    "physicalLayerCellIdGroup",
    "physicalLayerSubCellId",
    "pLMNId_mcc",
    "pLMNId_mnc",
    "administrativeState",
    "ExternalGNodeBFunction",
    "Regional",
    "Tipo_Freq"
]

missing_volta7= missing_volta7[colunas]
# Exibir o resultado
missing_volta7.head()

In [ ]:
missing_volta7.to_excel('C:/Camilao/Missing_GUtranCellRelation_volta.xlsx',index=None)

print('Done!!!')